In [ ]:
# Job configuration and helpers
import codecs
import os
import select
import shutil
import subprocess
import sys
import time
from pathlib import Path

BRANCH = 'feature/sota-baselines-bcic2a'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
TARGET_SUBJECTS = list(range(1, 5))
JOB_TAG = 's01_s09'

REPO_PATH = Path(f"/kaggle/working/bft-zhou2016-{JOB_TAG}")
MNE_DATA = Path("/kaggle/working/mne_data")
TRAIN_LOG = Path(f"/kaggle/working/bft_zhou2016_{JOB_TAG}.log")
RESULT_ARCHIVE = Path(f"/kaggle/working/bft_zhou2016_{JOB_TAG}_results")


def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print("+", " ".join(command), flush=True)
    if not stream:
        subprocess.run(
            command,
            cwd=str(cwd) if cwd else None,
            env=env,
            check=True,
        )
        return

    process = subprocess.Popen(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    started_at = time.monotonic()
    with open(log_path, "w", encoding="utf-8") as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end="", flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                elapsed = (time.monotonic() - started_at) / 60
                heartbeat = f"[Kaggle heartbeat] running | elapsed={elapsed:.1f}m\n"
                print(heartbeat, end="", flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break

        remaining = decoder.decode(b"", final=True)
        if remaining:
            print(remaining, end="", flush=True)
            log_file.write(remaining)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Training exited with code {return_code}. Full log: {log_path}"
        )

# Clone the exact source-only best-validation branch and install its pinned environment
run([sys.executable, "-m", "pip", "install", "-q", "uv"])

if (REPO_PATH / ".git").is_dir():
    run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_PATH)
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_PATH)
    run(["git", "checkout", "--force", BRANCH], cwd=REPO_PATH)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_PATH)
else:
    run([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        REPO_URL, REPO_PATH,
    ])

run(["git", "log", "-1", "--oneline"], cwd=REPO_PATH)

# Fail early if this is not the authors' implementation under the paper protocol.
runner_text = (REPO_PATH / "tta_baselines.py").read_text(encoding="utf-8")
assert "best source-validation accuracy" in runner_text
assert "zhou2016" in runner_text and "zhou2016:" in (REPO_PATH / "configs/tta_baselines.yaml").read_text(encoding="utf-8")
tta_text = (REPO_PATH / "utils/deeptransfer_tta.py").read_text(encoding="utf-8")
assert "def BFT_func(X_stream, model, ranker, args, device)" in tta_text
config_text = (REPO_PATH / "configs/tta_baselines.yaml").read_text(encoding="utf-8")
print("Verified: authors' BFT code, chronological LOSO, unlabeled target stream.")

UV = shutil.which("uv") or "uv"
run([UV, "venv", "--clear", "--python", "3.10", ".venv"], cwd=REPO_PATH)
PYTHON = REPO_PATH / ".venv/bin/python"
run([
    UV, "pip", "install", "--python", PYTHON,
    "torch==2.7.1", "torchvision==0.22.1",
    "--index-url", "https://download.pytorch.org/whl/cu126",
])
run([UV, "pip", "install", "--python", PYTHON, "-r", "requirements.txt"], cwd=REPO_PATH)

# Preserve method/data defaults; restrict targets.
override = (
    "import yaml; from pathlib import Path; "
    "p=Path('configs/tta_baselines.yaml'); "
    "c=yaml.safe_load(p.read_text()); "
    f"c['subject_ids']={TARGET_SUBJECTS!r}; "
    "p.write_text(yaml.safe_dump(c, sort_keys=False))"
)
run([PYTHON, "-c", override], cwd=REPO_PATH)

run(["nvidia-smi"])
run([
    PYTHON, "-c",
    "import torch; print('PyTorch:', torch.__version__); "
    "print('CUDA:', torch.cuda.is_available()); "
    "assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'; "
    "print('GPU:', torch.cuda.get_device_name(0))",
])

# Train only this job's LOSO target subjects and export its results
MNE_DATA.mkdir(parents=True, exist_ok=True)

# Zhou2016 is hosted on figshare, which intermittently answers Kaggle with
# HTTP 403. MOABB skips its own download when data.zip is already in place,
# so fetch it here with retries, or copy it from an attached Kaggle dataset.
def prefetch_zhou2016(mne_data):
    import glob, shutil, urllib.request, zipfile
    base = Path(mne_data) / 'MNE-zhou-2016'
    if (base / 'subject_1').is_dir() or (base / 'data').is_dir():
        print('Zhou2016 already extracted at', base, flush=True)
        return
    base.mkdir(parents=True, exist_ok=True)
    target = base / 'data.zip'

    def valid(path):
        try:
            with zipfile.ZipFile(path) as zf:
                return any(name.endswith('S1_1A.cnt') for name in zf.namelist())
        except (zipfile.BadZipFile, OSError):
            return False

    if target.exists() and valid(target):
        print('Zhou2016 data.zip already present:', target, flush=True)
        return
    for candidate in sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True)):
        if valid(candidate):
            shutil.copyfile(candidate, target)
            print('Zhou2016 data.zip copied from Kaggle input:', candidate, flush=True)
            return
    urls = ['https://ndownloader.figshare.com/files/3662952',
            'https://api.figshare.com/v2/file/download/3662952']
    # A browser user agent can receive figshare's bot-check page instead of
    # the file, so try plain client agents first.
    agents = ['python-requests/2.34.2', 'curl/8.5.0',
              'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) '
              'Chrome/126.0 Safari/537.36']
    last_error = None
    for attempt in range(1, 7):
        for url in urls:
            agent = agents[(attempt - 1) % len(agents)]
            try:
                request = urllib.request.Request(url, headers={'User-Agent': agent})
                with urllib.request.urlopen(request, timeout=120) as response, open(target, 'wb') as out:
                    shutil.copyfileobj(response, out)
                if valid(target):
                    print(f'Zhou2016 data.zip downloaded ({target.stat().st_size / 1e6:.1f} MB) '
                          f'from {url} on attempt {attempt}.', flush=True)
                    return
                last_error = 'downloaded file is not the Zhou2016 archive'
            except Exception as exc:
                last_error = exc
            print(f'Zhou2016 download attempt {attempt} via {url} failed: {last_error}', flush=True)
        time.sleep(min(30 * attempt, 120))
    raise RuntimeError(
        'Could not obtain Zhou2016 from figshare (last error: ' + str(last_error) + '). '
        'Download https://ndownloader.figshare.com/files/3662952 (data.zip) in a browser, '
        'upload it as a Kaggle dataset, attach it via Add Input, and rerun.')


prefetch_zhou2016(MNE_DATA)

# A rerun in the same draft session must not mix old and new result folders.
results_dir = REPO_PATH / "results"
if results_dir.exists():
    shutil.rmtree(results_dir)

environment = os.environ.copy()
environment.update({
    "PYTHONUNBUFFERED": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "MPLBACKEND": "Agg",
    "MNE_DATA": str(MNE_DATA),
    "MNE_DATASETS_BNCI_PATH": str(MNE_DATA),
    "MNE_DATASETS_ZHOU2016_PATH": str(MNE_DATA),
})

print(
    "===== ZHOU2016 BFT CHRONOLOGICAL LOSO | "
    f"TARGETS={TARGET_SUBJECTS} =====",
    flush=True,
)
print("Live log:", TRAIN_LOG, flush=True)

run([
    PYTHON, "-u", "tta_baselines.py",
    "--method", "bft",
    "--dataset", "zhou2016",
    "--gpu_id", "0",
], cwd=REPO_PATH, env=environment, stream=True, log_path=TRAIN_LOG)

archive = shutil.make_archive(
    str(RESULT_ARCHIVE),
    "zip",
    root_dir=REPO_PATH,
    base_dir="results",
)
print("Completed targets:", TARGET_SUBJECTS)
print("Kaggle output archive:", archive)
print("Kaggle output log:", TRAIN_LOG)
